# 03 – Transformation Pipeline (Issue #7)

Notebook này minh họa cách sử dụng `src/cleaning_pipeline.py` – phần độc lập của Issue #7.

**Lưu ý:** #5/#6 chưa hoàn thành. Notebook này dùng **fixture nhỏ** để demo.
Khi #5/#6 merge xong, chỉ cần thay fixture bằng `pd.read_parquet('data/interim/...')`.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42  # Hạt giống cố định theo quy ước dự án
np.random.seed(RANDOM_STATE)

# Notebook chạy với cwd = notebooks/, nên thư mục gốc repo là "..".
# sys.path phải trỏ tới GỐC REPO để import được package `src.*` — trỏ vào
# `src/` rồi import `cleaning_pipeline` không tôn trọng cấu trúc package.
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.cleaning_pipeline import (  # noqa: E402
    add_cyclical_time_features,
    chronological_split,
    build_preprocessing_pipeline,
    fit_pipeline_on_train,
    transform_with_pipeline,
    evaluate_log1p_transform,
    merge_air_weather,
    export_to_parquet,
    freeze_dataset,
    freeze_dataset_with_audit,
    run_preprocessing_audit,
    validate_no_leakage,
    NUMERIC_FEATURES,
    CYCLICAL_FEATURES,
)

## 1. Fixture nhỏ (demo – không phải dữ liệu thật)

Tạo DataFrame mẫu để test pipeline. Khi #5/#6 xong, thay bằng data thật.

In [2]:
# Fixture: 100 giờ, 1 trạm, có pm25 + số biến thời tiết
rng = np.random.RandomState(42)
n = 100

df_air = pd.DataFrame({
    'timestamp': pd.date_range('2024-01-01', periods=n, freq='h'),
    'station_id': 'ST01',
    'pm25': rng.lognormal(mean=3.5, sigma=0.8, size=n),
    'pm10': rng.lognormal(mean=4.0, sigma=0.7, size=n),
})

df_weather = pd.DataFrame({
    'timestamp': pd.date_range('2024-01-01', periods=n, freq='h'),
    'station_id': 'ST01',
    'temperature': rng.normal(25, 5, n),
    'relative_humidity': rng.uniform(40, 95, n),
    'wind_speed': rng.uniform(0, 10, n),
    'wind_direction': rng.uniform(0, 360, n),
    'precipitation': rng.exponential(2, n),
    'surface_pressure': rng.normal(1013, 5, n),
})

print(f'Air: {df_air.shape}, Weather: {df_weather.shape}')
df_air.head(3)

Air: (100, 4), Weather: (100, 8)


,timestamp,station_id,pm25,pm10
0,2024-01-01 00:00:00,ST01,49.272757,20.271997
1,2024-01-01 01:00:00,ST01,29.647819,40.672341
2,2024-01-01 02:00:00,ST01,55.598200,42.952714


## 2. Merge với row-explosion protection

In [3]:
df_merged = merge_air_weather(df_air, df_weather)
print(f'Merged: {df_merged.shape}')
assert len(df_merged) <= len(df_air), 'Row Explosion!'
df_merged.head(3)

Merged: (100, 10)


,timestamp,station_id,pm25,pm10,temperature,relative_humidity,wind_speed,wind_direction,precipitation,surface_pressure
0,2024-01-01 00:00:00,ST01,49.272757,20.271997,26.788937,55.511901,0.454464,27.310798,1.695455,1004.106399
1,2024-01-01 01:00:00,ST01,29.647819,40.672341,27.803923,49.759175,2.809632,46.396700,0.656950,1020.480222
2,2024-01-01 02:00:00,ST01,55.598200,42.952714,30.415256,81.283811,9.504115,46.096502,2.934948,1016.271828


## 3. Cyclical time features

In [4]:
df_merged = add_cyclical_time_features(df_merged)
print('Cyclical features added:', CYCLICAL_FEATURES)
df_merged[['timestamp', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']].head(3)

Cyclical features added: ['hour_sin', 'hour_cos', 'month_sin', 'month_cos']


,timestamp,hour_sin,hour_cos,month_sin,month_cos
0,2024-01-01 00:00:00,0.000000,1.000000,0.5,0.866025
1,2024-01-01 01:00:00,0.258819,0.965926,0.5,0.866025
2,2024-01-01 02:00:00,0.500000,0.866025,0.5,0.866025


## 4. Dataset freeze (ghi nhận metadata)

In [5]:
freeze_info = freeze_dataset(df_merged)
print(f'Rows: {freeze_info.row_count}')
print(f'Span: {freeze_info.timestamp_min} → {freeze_info.timestamp_max}')
print(f'Stations: {freeze_info.station_count}')

Rows: 100
Span: 2024-01-01 00:00:00 → 2024-01-05 03:00:00
Stations: 1


## 5. Chronological split (không random)

Split point sẽ được quyết định sau khi #5/#6 hoàn thành.
Ở đây demo với split tại 70% thời gian.

In [6]:
# Demo split point – sẽ thay bằng giá trị thực tế sau #5/#6
split_ts = df_merged['timestamp'].quantile(0.7)
print(f'Split timestamp (demo): {split_ts}')

train_df, test_df = chronological_split(df_merged, split_ts)
print(f'Train: {len(train_df)}, Test: {len(test_df)}')

# Validation cứng
assert train_df['timestamp'].max() < test_df['timestamp'].min()
print('✓ No temporal leakage')

Split timestamp (demo): 2024-01-03 21:18:00
Train: 70, Test: 30
✓ No temporal leakage


## 6. Build & fit Pipeline (CHỈ trên Train)

In [7]:
# Chọn features có trong fixture
numeric_feats = [c for c in NUMERIC_FEATURES if c in train_df.columns]
cyc_feats = [c for c in CYCLICAL_FEATURES if c in train_df.columns]

print(f'Numeric: {numeric_feats}')
print(f'Cyclical: {cyc_feats}')

pipeline = build_preprocessing_pipeline(
    numeric_features=numeric_feats,
    cyclical_features=cyc_feats,
)

X_train = train_df[numeric_feats + cyc_feats]
X_test = test_df[numeric_feats + cyc_feats]

# Fit STRICTLY on Train
pipeline = fit_pipeline_on_train(pipeline, X_train)
print('✓ Pipeline fitted on Train only')

Numeric: ['pm25', 'pm10', 'temperature', 'relative_humidity', 'wind_speed', 'wind_direction', 'precipitation', 'surface_pressure']
Cyclical: ['hour_sin', 'hour_cos', 'month_sin', 'month_cos']
✓ Pipeline fitted on Train only


## 7. Transform Train & Test

In [8]:
X_train_transformed = transform_with_pipeline(pipeline, X_train)
X_test_transformed = transform_with_pipeline(pipeline, X_test)

print(f'X_train_transformed: {X_train_transformed.shape}')
print(f'X_test_transformed: {X_test_transformed.shape}')
print('✓ Test transformed using Train-fitted parameters (no leakage)')

X_train_transformed: (70, 12)
X_test_transformed: (30, 12)
✓ Test transformed using Train-fitted parameters (no leakage)


## 8. Target transformation diagnostic (log1p candidate)

CHỉ diagnostic – KHÔNG kết luận log1p tốt hơn.

In [9]:
y_train = train_df['pm25']
diag = evaluate_log1p_transform(y_train)

print(f'Original skew: {diag["original_skew"]:.3f}')
print(f'log1p skew:    {diag["log1p_skew"]:.3f}')
print(f'Original kurtosis: {diag["original_kurtosis"]:.3f}')
print(f'log1p kurtosis:    {diag["log1p_kurtosis"]:.3f}')
print('\n→ Kết luận cuối cùng sẽ dựa trên dữ liệu thực tế sau #5/#6.')

Original skew: 1.611
log1p skew:    0.085
Original kurtosis: 2.738
log1p kurtosis:    -0.534

→ Kết luận cuối cùng sẽ dựa trên dữ liệu thực tế sau #5/#6.


## 9. Integration với Issue #5 (audit trước preprocessing)

Reuse `audit_dataframe` và `audit_six_dimensions` từ `data_quality.py`.

In [10]:
# Chạy audit từ #5 trên dataset
audit_results = run_preprocessing_audit(df_merged, dataset_name='demo_dataset')

print('Summary table:')
display(audit_results['summary_table'])

print('\nSix dimensions keys:', list(audit_results['six_dimensions']['dimensions'].keys()))

Summary table:


,column_name,dtype,total_count,missing_count,missing_percentage,unique_count,min,max,q25,q50,q75
0,timestamp,datetime64[ns],100,0,0.0,100,2024-01-01 00:00:00,2024-01-05 03:00:00,NaN,NaN,NaN
1,station_id,object,100,0,0.0,1,ST01,ST01,NaN,NaN,NaN
2,pm25,float64,100,0,0.0,100,4.072284,145.739757,20.476452,2.991847e+01,45.862804
3,pm10,float64,100,0,0.0,100,14.251471,366.543944,31.063889,5.791256e+01,79.592061
4,temperature,float64,100,0,0.0,100,8.793663,44.263657,21.722782,2.548848e+01,28.522187
5,relative_humidity,float64,100,0,0.0,100,40.596071,94.613064,56.023486,7.034304e+01,83.746613
6,wind_speed,float64,100,0,0.0,100,0.145447,9.997177,3.433795,5.581848e+00,8.264851
7,wind_direction,float64,100,0,0.0,100,7.225631,355.14549,62.319659,1.319730e+02,227.211832
8,precipitation,float64,100,0,0.0,100,0.068215,9.202403,0.577960,1.407088e+00,2.779199
9,surface_pressure,float64,100,0,0.0,100,999.515567,1025.866799,1009.467654,1.012547e+03,1016.557348



Six dimensions keys: ['completeness', 'accuracy', 'consistency', 'validity', 'uniqueness', 'timeliness']


## 10. Leakage validation

Kiểm tra pipeline không fit trên Test.

In [11]:
# Validate không có leakage
validate_no_leakage(pipeline, X_train, X_test)
print('✓ Pipeline validated: fitted on Train only, no leakage')

✓ Pipeline validated: fitted on Train only, no leakage


## 11. Freeze dataset with audit

Freeze dataset và ghi nhận audit results.

In [12]:
# Freeze với audit
freeze_info, audit = freeze_dataset_with_audit(df_merged, run_audit=True)

print(f'Rows: {freeze_info.row_count}')
print(f'Span: {freeze_info.timestamp_min} → {freeze_info.timestamp_max}')
print(f'Stations: {freeze_info.station_count}')
print(f'Audit recorded: {"audit" in freeze_info.additional_info}')

Rows: 100
Span: 2024-01-01 00:00:00 → 2024-01-05 03:00:00
Stations: 1
Audit recorded: True


## 12. Export Parquet (snappy)

Khi #5/#6 xong, export dataset thật ra `data/processed/air_pollution_final.parquet`.

In [13]:
# Demo export – khi #5/#6 xong, thay bằng:
# export_to_parquet(df_merged, 'data/processed/air_pollution_final.parquet')

import tempfile
with tempfile.NamedTemporaryFile(suffix='.parquet', delete=False) as f:
    tmp_path = f.name

export_to_parquet(df_merged, tmp_path)
print(f'✓ Exported and validated: {tmp_path}')

# Cleanup
import os
os.unlink(tmp_path)

✓ Exported and validated: C:\Users\Admin\AppData\Local\Temp\tmpeulcdjcp.parquet


## Tổng kết

| Bước | Trạng thái |
|------|------------|
| Cyclical features | ✓ |
| Merge + row explosion protection | ✓ |
| Dataset freeze helper | ✓ |
| Chronological split | ✓ |
| Pipeline fit on Train only | ✓ |
| Transform Train/Test | ✓ |
| log1p diagnostic | ✓ |
| Parquet export | ✓ |
| Integration với #5 (audit) | ✓ |
| Leakage validation | ✓ |
| Freeze with audit | ✓ |

**Chờ #6:** thay fixture bằng data thật, quyết định split point, kết luận log1p.